# PBISC — floor를 real Top10 marker로 직접 annotation하면 real과 가까워지는가?
**설계**
1. real A(저자 라벨)에서 계층적 Top10 marker: 6-type vs 나머지, fine subset vs 같은 6-type 내 sibling (Wilcoxon, log2FC>0.5, FDR<0.05, pct_in>pct_out, RP/MT 유전자 제외).
2. floor와 real B(다른 환자, 저자 라벨 보유, floor depth로 6-type별 binomial thinning)를 **각자의 PCA로 de novo Leiden** (L1 res 1.0 → 6-type, L2 6-type 내 res 2.0 → fine), cluster 평균 `score_genes`의 argmax로 annotation. cell-level argmax는 민감도 분석.
3. 비교 기준: real A. 지표 = subset 비율 log2 ratio, 비-marker HVG에서의 identity profile r (subset 평균 − dataset 평균), marker contrast ratio.
4. **같은 marker 자(ruler)를 real B에 적용**해 저자 라벨 회복력(F1)을 측정 → F1≥0.5인 'marker-resolvable' subset에서 공정 비교.

Step 1–2는 스크립트(`s1_markers.py`, `s2_annotate.py`, 약 15분)로 실행했고 아래에 코드를 첨부합니다. 이 notebook은 Step 3 이후를 재실행합니다.

### `s1_markers.py`
```python
# Step 1: hierarchical Top10 markers from real A (author labels)
import numpy as np, pandas as pd, anndata as ad, scanpy as sc, pickle, warnings; warnings.filterwarnings("ignore")
B="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/big/"
MINC=20
def top10(a,key,groups,ref="rest"):
    sc.tl.rank_genes_groups(a,key,groups=groups,reference=ref,method="wilcoxon",pts=True,use_raw=False)
    out={}
    for g in groups:
        d=sc.get.rank_genes_groups_df(a,g)
        d=d[~d.names.str.match(r"^(RP[SL]|MT-)")]  # drop ribosomal / mito genes
        ok=d[(d.logfoldchanges>0.5)&(d.pvals_adj<0.05)]
        if "pct_nz_group" in d: ok=ok[ok.pct_nz_group>ok.pct_nz_reference]
        genes=ok.names.tolist()[:10]
        if len(genes)<10: genes+= [x for x in d.names if x not in genes][:10-len(genes)]
        out[g]=genes
    return out
M={}
for coh in ["sepsis","covid"]:
    a=ad.read_h5ad(B+f"{coh}_ref.h5ad"); a.obs["six_type"]=a.obs.six_type.astype(str); a.obs["fine"]=a.obs.fine.astype(str)
    sc.pp.normalize_total(a,1e4); sc.pp.log1p(a)
    six=[s for s,n in a.obs.six_type.value_counts().items() if n>=MINC]
    L1=top10(a,"six_type",six)
    L2={}
    for s in six:
        b=a[a.obs.six_type==s].copy()
        fines=[f for f,n in b.obs.fine.value_counts().items() if n>=MINC]
        if len(fines)<2: L2[s]={f:L1[s] for f in fines}; continue
        L2[s]=top10(b,"fine",fines)
    M[coh]=dict(L1=L1,L2=L2)
    print(coh,{s:len(v) for s,v in L2.items()})
    for s in L2:
        for f,g in L2[s].items(): print("  ",s,"|",f,":",",".join(g))
pickle.dump(M,open("markerann/markers.pkl","wb"))
rows=[dict(cohort=c,level="six_type",six_type=s,subset=s,markers=",".join(g)) for c in M for s,g in M[c]["L1"].items()]
rows+=[dict(cohort=c,level="fine",six_type=s,subset=f,markers=",".join(g)) for c in M for s in M[c]["L2"] for f,g in M[c]["L2"][s].items()]
pd.DataFrame(rows).to_csv("markerann/top10_markers_realA.csv",index=False)

```

### `s2_annotate.py`
```python
# Step 2: de novo clustering + Top10-marker annotation (two-level) of floor and real B
import numpy as np, pandas as pd, anndata as ad, scanpy as sc, pickle, sys, warnings; warnings.filterwarnings("ignore")
import scipy.sparse as sp
B="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/"
M=pickle.load(open("markerann/markers.pkl","rb"))
rng=np.random.default_rng(0)
def thin(X,p):
    X=sp.csr_matrix(X).astype(np.int64); X.sort_indices()
    rows=np.repeat(np.arange(X.shape[0]),np.diff(X.indptr))
    X.data=rng.binomial(X.data,p[rows]); X.eliminate_zeros(); return X.astype(np.float32)
def cluster(a,res,npc):
    b=a.copy(); sc.pp.highly_variable_genes(b,n_top_genes=2000,flavor="seurat")
    b=b[:,b.var.highly_variable].copy(); sc.pp.scale(b,max_value=10)
    sc.tl.pca(b,n_comps=min(npc,b.n_obs-1)); sc.pp.neighbors(b,n_neighbors=15)
    sc.tl.leiden(b,resolution=res,key_added="cl",flavor="igraph",n_iterations=2,directed=False)
    return b.obs.cl.astype(str).values
def scores(a,sets):
    S={}
    for k,g in sets.items():
        g=[x for x in g if x in a.var_names]
        sc.tl.score_genes(a,g,score_name="_s",random_state=0,use_raw=False); S[k]=a.obs["_s"].values.copy()
    return pd.DataFrame(S,index=a.obs_names)
def annotate(a,coh):
    L1,L2=M[coh]["L1"],M[coh]["L2"]
    a.obs["cl1"]=cluster(a,1.0,30)
    S1=scores(a,L1)
    cm=S1.groupby(a.obs.cl1.values).mean()
    a.obs["six_mk_cl"]=a.obs.cl1.map(cm.idxmax(1)).values
    a.obs["six_mk_cell"]=S1.idxmax(1).values
    a.obs["fine_mk_cl"]=""; a.obs["fine_mk_cell"]=""; a.obs["cl2"]=""
    for s in L1:
        m=(a.obs.six_mk_cl==s).values
        if m.sum()>=30:
            b=a[m].copy(); b.obs["cl2"]=cluster(b,2.0,20)
            S2=scores(b,L2[s]); cm2=S2.groupby(b.obs.cl2.values).mean()
            a.obs.loc[m,"fine_mk_cl"]=b.obs.cl2.map(cm2.idxmax(1)).values
            a.obs.loc[m,"cl2"]=(s+"_"+b.obs.cl2.astype(str)).values
        mc=(a.obs.six_mk_cell==s).values
        if mc.sum():
            b=a[mc].copy(); S2=scores(b,L2[s]); a.obs.loc[mc,"fine_mk_cell"]=S2.idxmax(1).values
    return a
for coh in ["sepsis","covid"]:
    f=ad.read_h5ad(B+f"big/{coh}_floor.h5ad")
    j=ad.read_h5ad(B+f"big_joint/{coh}_joint_floor.h5ad",backed="r"); jf=j.obs[j.obs.source=="floor"]
    f.obs["six_knn"]=jf.six_tr.astype(str).values; f.obs["fine_knn"]=jf.fine_tr.astype(str).values
    u=ad.read_h5ad(B+f"big/{coh}_realU.h5ad")
    flib=pd.Series(np.asarray(f.X.sum(1)).ravel()).groupby(f.obs.six_knn.values).median()
    ulib=pd.Series(np.asarray(u.X.sum(1)).ravel()).groupby(u.obs.six_type.astype(str).values).median()
    p=(flib/ulib).clip(upper=1).reindex(u.obs.six_type.astype(str).values).fillna(1).values
    print(coh,"thinning p per six:",(flib/ulib).round(2).to_dict())
    ut=u.copy(); ut.X=thin(u.X,p)
    for name,a in [("floor",f),("realB_thin",ut),("realB_full",u)]:
        a.obs_names_make_unique()
        sc.pp.normalize_total(a,1e4); sc.pp.log1p(a)
        a=annotate(a,coh)
        cols=[c for c in ["sample","patient","group","six_type","fine","six_knn","fine_knn","cl1","cl2","six_mk_cl","fine_mk_cl","six_mk_cell","fine_mk_cell"] if c in a.obs]
        a.obs[cols].to_csv(f"markerann/{coh}_{name}_labels.csv.gz")
        if name!="realB_full": a.write_h5ad(f"markerann/{coh}_{name}_annot.h5ad",compression="gzip")
        print(coh,name,"n clusters L1",a.obs.cl1.nunique(),"L2",a.obs.cl2.nunique(),flush=True)

```

In [ ]:
import os; os.chdir('/home/claude/pbisc')
import pandas as pd
pd.set_option('display.width',250); pd.set_option('display.max_columns',30)
mk=pd.read_csv('markerann/top10_markers_realA.csv'); mk.head(20)

## Step 3 — 비율·발현 유사도 계산

In [ ]:
exec(open('markerann/s3_compare.py').read())

## Step 4 — real B에서의 marker 자 해상도 + key subset 표

In [ ]:
exec(open('markerann/s4_resolvable.py').read())

## 그림

In [ ]:
exec(open('markerann/fig_markerann.py').read())
from IPython.display import Image, display
for f in ['fig1_ruler_share_identity','fig2_key_subsets_share']: display(Image(f'markerann/{f}.png',width=1100))

## floor에서 marker로 '희귀 subset'이라 불린 세포의 kNN 정체

In [ ]:
for coh,s in [("sepsis","Macrophage"),("sepsis","pDC"),("covid","CD16mono_C1"),("covid","DC1"),("covid","MAIT")]:
    F=pd.read_csv(f"markerann/{coh}_floor_labels.csv.gz",index_col=0); m=F.fine_mk_cl==s
    print(coh,s,"n=",m.sum(),"| kNN labels:",F[m].fine_knn.value_counts().head(4).to_dict())